# Task 4: Machine Learning Modeling

This notebook builds predictive models to help ACIS optimize their insurance pricing and risk assessment. We'll develop models for:
1. **Claim Prediction**: Predicting whether a policy will have claims (binary classification)
2. **Claim Severity**: Predicting the amount of claims (regression)
3. **Premium Optimization**: Suggesting optimal premium amounts based on risk factors

In [ ]:
import sys
sys.path.append('../')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import load_data

sns.set_theme(style="whitegrid")

## 1. Data Preparation

In [ ]:
# Load the cleaned data
print("Loading cleaned insurance data...")
df = load_data('../data/insurance_data_cleaned.csv')

# Create target variables
df['HasClaim'] = (df['TotalClaims'] > 0).astype(int)
df['ClaimAmount'] = df['TotalClaims']
df['PremiumToClaimRatio'] = df['TotalPremium'] / (df['TotalClaims'] + 1)  # Add 1 to avoid division by zero

print(f"Dataset shape: {df.shape}")
print(f"Claim rate: {df['HasClaim'].mean():.2%}")
print(f"Average claim amount: R{df[df['TotalClaims'] > 0]['TotalClaims'].mean():.2f}")

In [ ]:
# Feature engineering
def prepare_features(df):
    """Prepare features for modeling"""
    df_model = df.copy()
    
    # Handle missing values
    df_model['Gender'] = df_model['Gender'].fillna('Unknown')
    df_model['MaritalStatus'] = df_model['MaritalStatus'].fillna('Unknown')
    
    # Create new features
    df_model['VehicleAge'] = 2015 - df_model['RegistrationYear']  # Assuming data is from 2015
    df_model['IsNewVehicle'] = (df_model['VehicleAge'] <= 2).astype(int)
    df_model['HighValueVehicle'] = (df_model['SumInsured'] > df_model['SumInsured'].quantile(0.75)).astype(int)
    
    # Encode categorical variables
    categorical_cols = ['Province', 'Gender', 'MaritalStatus', 'VehicleType', 'make', 'Model']
    
    for col in categorical_cols:
        if col in df_model.columns:
            le = LabelEncoder()
            df_model[f'{col}_encoded'] = le.fit_transform(df_model[col].astype(str))
    
    return df_model

df_prepared = prepare_features(df)
print("Feature engineering completed.")

## 2. Model 1: Claim Prediction (Binary Classification)

In [ ]:
# Select features for claim prediction
feature_cols = [
    'Province_encoded', 'Gender_encoded', 'MaritalStatus_encoded',
    'SumInsured', 'CustomValueEstimate', 'VehicleAge', 'IsNewVehicle', 'HighValueVehicle',
    'PostalCode', 'Cylinders', 'cubiccapacity', 'kilowatts', 'NumberOfDoors'
]

# Filter available columns
available_features = [col for col in feature_cols if col in df_prepared.columns]
print(f"Using {len(available_features)} features for modeling")

X = df_prepared[available_features].fillna(0)
y_claim = df_prepared['HasClaim']

# Split the data
X_train, X_test, y_train_claim, y_test_claim = train_test_split(
    X, y_claim, test_size=0.2, random_state=42, stratify=y_claim
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

In [ ]:
# Train multiple models for claim prediction
models_claim = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

claim_results = {}

for name, model in models_claim.items():
    print(f"\nTraining {name}...")
    
    # Train the model
    model.fit(X_train, y_train_claim)
    
    # Make predictions
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
    
    # Calculate metrics
    accuracy = model.score(X_test, y_test_claim)
    auc = roc_auc_score(y_test_claim, y_pred_proba) if y_pred_proba is not None else None
    
    claim_results[name] = {
        'model': model,
        'accuracy': accuracy,
        'auc': auc,
        'predictions': y_pred
    }
    
    print(f"Accuracy: {accuracy:.4f}")
    if auc:
        print(f"AUC: {auc:.4f}")
    print("Classification Report:")
    print(classification_report(y_test_claim, y_pred))

## 3. Model 2: Claim Severity Prediction (Regression)

In [ ]:
# Prepare data for severity modeling (only policies with claims)
df_claims = df_prepared[df_prepared['TotalClaims'] > 0].copy()
print(f"Policies with claims: {len(df_claims)}")

if len(df_claims) > 100:  # Only proceed if we have enough data
    X_severity = df_claims[available_features].fillna(0)
    y_severity = df_claims['ClaimAmount']
    
    # Split the data
    X_train_sev, X_test_sev, y_train_sev, y_test_sev = train_test_split(
        X_severity, y_severity, test_size=0.2, random_state=42
    )
    
    # Train severity models
    models_severity = {
        'Linear Regression': LinearRegression(),
        'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42)
    }
    
    severity_results = {}
    
    for name, model in models_severity.items():
        print(f"\nTraining {name} for severity prediction...")
        
        # Train the model
        model.fit(X_train_sev, y_train_sev)
        
        # Make predictions
        y_pred_sev = model.predict(X_test_sev)
        
        # Calculate metrics
        mse = mean_squared_error(y_test_sev, y_pred_sev)
        r2 = r2_score(y_test_sev, y_pred_sev)
        
        severity_results[name] = {
            'model': model,
            'mse': mse,
            'r2': r2,
            'predictions': y_pred_sev
        }
        
        print(f"MSE: {mse:.2f}")
        print(f"R²: {r2:.4f}")
        print(f"RMSE: {np.sqrt(mse):.2f}")
else:
    print("Not enough claims data for severity modeling")
    severity_results = {}

## 4. Feature Importance Analysis

In [ ]:
# Get feature importance from Random Forest model
if 'Random Forest' in claim_results:
    rf_model = claim_results['Random Forest']['model']
    feature_importance = pd.DataFrame({
        'feature': available_features,
        'importance': rf_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    # Plot feature importance
    plt.figure(figsize=(10, 6))
    sns.barplot(data=feature_importance.head(10), x='importance', y='feature')
    plt.title('Top 10 Feature Importance for Claim Prediction')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.savefig('../reports/feature_importance.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("Top 10 Most Important Features:")
    print(feature_importance.head(10))

## 5. Model Performance Visualization

In [ ]:
# Create performance comparison
performance_data = []
for name, results in claim_results.items():
    performance_data.append({
        'Model': name,
        'Accuracy': results['accuracy'],
        'AUC': results['auc'] if results['auc'] else 0
    })

performance_df = pd.DataFrame(performance_data)

# Plot model comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Accuracy comparison
sns.barplot(data=performance_df, x='Model', y='Accuracy', ax=ax1)
ax1.set_title('Model Accuracy Comparison')
ax1.set_ylim(0, 1)

# AUC comparison
auc_data = performance_df[performance_df['AUC'] > 0]
if not auc_data.empty:
    sns.barplot(data=auc_data, x='Model', y='AUC', ax=ax2)
    ax2.set_title('Model AUC Comparison')
    ax2.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('../reports/model_performance.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nModel Performance Summary:")
print(performance_df)

## 6. Risk Scoring and Premium Recommendations

In [ ]:
# Create a risk scoring function
def calculate_risk_score(model, X_data):
    """Calculate risk scores using the best performing model"""
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(X_data)[:, 1]  # Probability of claim
    else:
        return model.predict(X_data)

# Use the best performing model (highest AUC)
best_model_name = max(claim_results.keys(), key=lambda x: claim_results[x]['auc'] or 0)
best_model = claim_results[best_model_name]['model']

print(f"Best performing model: {best_model_name}")

# Calculate risk scores for test set
risk_scores = calculate_risk_score(best_model, X_test)

# Create risk categories
risk_categories = pd.cut(risk_scores, 
                        bins=[0, 0.1, 0.3, 0.7, 1.0], 
                        labels=['Low Risk', 'Medium Risk', 'High Risk', 'Very High Risk'])

# Create results dataframe
results_df = pd.DataFrame({
    'RiskScore': risk_scores,
    'RiskCategory': risk_categories,
    'ActualClaim': y_test_claim.values
})

# Analyze risk categories
risk_analysis = results_df.groupby('RiskCategory').agg({
    'RiskScore': ['count', 'mean'],
    'ActualClaim': 'mean'
}).round(4)

risk_analysis.columns = ['Count', 'Avg_Risk_Score', 'Actual_Claim_Rate']
print("\nRisk Category Analysis:")
print(risk_analysis)

In [ ]:
# Premium recommendation function
def recommend_premium_adjustment(risk_score, base_premium=1000):
    """Recommend premium adjustments based on risk score"""
    if risk_score < 0.1:
        return base_premium * 0.8  # 20% discount for low risk
    elif risk_score < 0.3:
        return base_premium * 0.9  # 10% discount for medium-low risk
    elif risk_score < 0.7:
        return base_premium * 1.0  # Standard premium
    else:
        return base_premium * 1.3  # 30% increase for high risk

# Apply premium recommendations
results_df['RecommendedPremium'] = results_df['RiskScore'].apply(recommend_premium_adjustment)

# Visualize risk distribution
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.hist(risk_scores, bins=30, alpha=0.7, edgecolor='black')
plt.xlabel('Risk Score')
plt.ylabel('Frequency')
plt.title('Distribution of Risk Scores')

plt.subplot(1, 2, 2)
risk_category_counts = results_df['RiskCategory'].value_counts()
plt.pie(risk_category_counts.values, labels=risk_category_counts.index, autopct='%1.1f%%')
plt.title('Risk Category Distribution')

plt.tight_layout()
plt.savefig('../reports/risk_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Model Validation and Business Impact

In [ ]:
# Calculate potential business impact
def calculate_business_impact(results_df, current_premium=1000):
    """Calculate the potential business impact of risk-based pricing"""
    
    # Current scenario (flat pricing)
    current_revenue = len(results_df) * current_premium
    
    # Proposed scenario (risk-based pricing)
    proposed_revenue = results_df['RecommendedPremium'].sum()
    
    # Calculate metrics by risk category
    impact_analysis = results_df.groupby('RiskCategory').agg({
        'RecommendedPremium': 'mean',
        'ActualClaim': 'mean',
        'RiskScore': 'count'
    }).round(2)
    
    impact_analysis.columns = ['Avg_Premium', 'Claim_Rate', 'Policy_Count']
    impact_analysis['Premium_vs_Current'] = (impact_analysis['Avg_Premium'] / current_premium - 1) * 100
    
    return {
        'current_revenue': current_revenue,
        'proposed_revenue': proposed_revenue,
        'revenue_change': proposed_revenue - current_revenue,
        'revenue_change_pct': (proposed_revenue / current_revenue - 1) * 100,
        'impact_by_category': impact_analysis
    }

business_impact = calculate_business_impact(results_df)

print("Business Impact Analysis:")
print(f"Current Revenue (Flat Pricing): R{business_impact['current_revenue']:,.2f}")
print(f"Proposed Revenue (Risk-Based): R{business_impact['proposed_revenue']:,.2f}")
print(f"Revenue Change: R{business_impact['revenue_change']:,.2f} ({business_impact['revenue_change_pct']:.1f}%)")
print("\nImpact by Risk Category:")
print(business_impact['impact_by_category'])

## 8. Model Deployment Recommendations

In [ ]:
# Save the best model and create deployment summary
import joblib

# Save the best model
model_path = '../models/best_claim_prediction_model.pkl'
joblib.dump(best_model, model_path)
print(f"Best model saved to: {model_path}")

# Create deployment summary
deployment_summary = {
    'model_type': best_model_name,
    'accuracy': claim_results[best_model_name]['accuracy'],
    'auc': claim_results[best_model_name]['auc'],
    'features_used': available_features,
    'risk_categories': {
        'Low Risk (0-10%)': 'Premium discount: 20%',
        'Medium Risk (10-30%)': 'Premium discount: 10%',
        'High Risk (30-70%)': 'Standard premium',
        'Very High Risk (70%+)': 'Premium increase: 30%'
    },
    'expected_revenue_impact': f"{business_impact['revenue_change_pct']:.1f}%"
}

print("\n" + "="*50)
print("MODEL DEPLOYMENT SUMMARY")
print("="*50)
print(f"Best Model: {deployment_summary['model_type']}")
print(f"Accuracy: {deployment_summary['accuracy']:.1%}")
print(f"AUC Score: {deployment_summary['auc']:.3f}")
print(f"Expected Revenue Impact: {deployment_summary['expected_revenue_impact']}")
print("\nRecommended Implementation:")
print("1. Deploy model for real-time risk scoring")
print("2. Implement risk-based premium adjustments")
print("3. Monitor model performance monthly")
print("4. Retrain model quarterly with new data")